# The lag against the in-estuary gauges, across sites

`02_lag_vs_gauges_site.ipynb` writes one row per window, judge, method and truth to
`results/lag_nb/<site>.csv` (Escalda, Wadden, Ems, Ferrol; Villaviciosa has no judge and its
file holds band lags). This notebook reads those files and prints the first lag table of the
pre-lock run: each site at its first window, the error of every method against the three
co-primary truths, (a) the best constant lag at the scene instants, τ_ab (the same with a free
gain) and (b) the M2 phase lag.

Pre-lock and descriptive: W3, the confirmatory window, is blinded and absent. The scene instants
are catalogue times, so the imagery lags are expected about 6.5 min low; nothing is corrected.
MAREA's parameters were developed on these sites and judges (home advantage).

## Parameters

In [ ]:
import os

OUT_DIR = os.environ.get("NB_OUT", "results/lag_nb")
SITES = ["escalda", "wadden", "ems", "ferrol"]
FIRST = {"escalda": "Y2023", "wadden": "W2", "ems": "Y2023", "ferrol": "Y2024"}   # the first lag table's windows

## 0 · Setup

In [ ]:
for _ in range(3):                                # this notebook lives in notebooks/validation/
    if os.path.isdir("pyintertidal"):
        break
    os.chdir("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NAMES = {"L01": "MAREA (m2a band lags)", "L17": "zero lag", "L14": "EOT20 at the gauge",
         "L04w": "XC wet share", "L04i": "XC median index", "L07": "LOOP", "L08": "GM20",
         "L09": "HD26-H", "L10": "HD26-R2", "L11": "PPL", "L12": "BT26-shift",
         "L18": "shallow-water celerity", "L20": "REF gauge network (never ranked)"}

## 1 · The per-judge rows

e = τ_method − τ_truth in minutes (circular for the phase-type lags), one row per window, judge,
method and truth.

In [ ]:
found = [s for s in SITES if os.path.exists(f"{OUT_DIR}/{s}.csv")]
points = pd.concat([pd.read_csv(f"{OUT_DIR}/{s}.csv") for s in found], ignore_index=True)
print(f"sites: {found} (missing: {sorted(set(SITES) - set(found))})")
points.groupby(["site", "window"]).agg(judges=("judge", "nunique"), locations=("location", "nunique"),
                                       methods=("method", "nunique"), rows=("e", "size"))

## 2 · The first lag table

Per site and window, method and truth: the gauges of one location are averaged first (the two
Terneuzen gauges are one location), then every location counts once: MAE = mean |e|, bias =
mean e, hit10 = % of locations within 10 min.

In [ ]:
loc = (points.assign(abs_e=points.e.abs())
       .groupby(["site", "window", "method", "truth", "location"])[["e", "abs_e"]].mean().reset_index())
errors = (loc.groupby(["site", "window", "method", "truth"])
          .agg(n_loc=("location", "size"), MAE=("abs_e", "mean"), bias=("e", "mean"),
               hit10=("abs_e", lambda v: 100 * np.mean(v <= 10))).reset_index())

for site in found:
    e = errors[(errors.site == site) & (errors.window == FIRST[site])]
    t = e.pivot_table(index="method", columns="truth", values=["MAE", "bias", "hit10"])
    t = t.reindex([m for m in NAMES if m in t.index]).swaplevel(axis=1)[["a", "ab", "b"]]
    t.insert(0, "n loc", e.groupby("method").n_loc.first())
    t.index = [NAMES[m] for m in t.index]
    print(f"\n{site} · {FIRST[site]}")
    display(t.round(1))

## 3 · Every window

MAE against truth (a) for every scored window (the Wadden has four).

In [ ]:
wide = errors[errors.truth == "a"].pivot_table(index="method", columns=["site", "window"], values="MAE")
wide = wide.reindex([m for m in NAMES if m in wide.index])
wide.index = [NAMES[m] for m in wide.index]
wide.round(1)

## 4 · Figure

MAE of every method at each site's first window, against truths (a) and (b).

In [ ]:
COLORS = {"escalda": "#0A7FA8", "wadden": "#B7791F", "ems": "#2E8B57", "ferrol": "#8E44AD"}
methods = [m for m in NAMES if m in set(errors.method)]
fig, axes = plt.subplots(1, 2, figsize=(13, 0.45 * len(methods) + 1.5), sharey=True)
for ax, T in zip(axes, ("a", "b")):
    for site in found:
        e = errors[(errors.site == site) & (errors.window == FIRST[site]) & (errors.truth == T)]
        e = e.set_index("method").reindex(methods)
        ax.plot(e.MAE, np.arange(len(methods)), "o", ms=7, color=COLORS[site],
                label=f"{site} · {FIRST[site]}")
    ax.set_yticks(np.arange(len(methods)), [NAMES[m] for m in methods])
    ax.axvline(10, color="0.5", lw=0.8, ls=":")
    ax.set_xlabel(f"MAE against truth ({T}) over the locations (min)")
    ax.grid(alpha=0.3, axis="x")
axes[0].invert_yaxis()                            # MAREA on top (the y axis is shared)
axes[0].legend(fontsize=8)
fig.suptitle("Lag against the in-estuary gauges: pre-lock, descriptive", x=0.01, ha="left")
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/02b_summary.png", dpi=130, bbox_inches="tight")